In [1]:
!pip install transformers torch tqdm

In [2]:
!pip install --upgrade transformers

In [3]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
df = pd.read_csv("../content/openalex_metadata_full.csv")

print("Total rows:", len(df))
print("Null abstracts:", df['abstract'].isna().sum())

Total rows: 2529
Null abstracts: 18


In [6]:
df = df.dropna(subset=["abstract"])
df = df[df["abstract"].str.strip() != ""]

print("Remaining rows after cleaning:", len(df))

Remaining rows after cleaning: 2511


In [7]:
paper_ids = df["global_paper_id"].tolist()
abstracts = df["abstract"].tolist()

In [8]:
## Load SciBERT
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
model = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")

model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31090, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [9]:
## Embedding Function
## SciBERT outputs token embeddings.
## We take mean pooled representation.
def get_embeddings(text_list, batch_size=16, max_length=512):
    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size)):
            batch = text_list[i:i+batch_size]

            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs = model(**inputs)

            # last hidden state
            last_hidden = outputs.last_hidden_state  # (batch, seq_len, hidden_dim)

            # mean pooling
            attention_mask = inputs["attention_mask"]
            mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()

            summed = torch.sum(last_hidden * mask_expanded, dim=1)
            summed_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

            mean_pooled = summed / summed_mask

            embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(embeddings)

In [10]:
## Generate Embeddings
abstract_embeddings = get_embeddings(abstracts, batch_size=16)

print("Embedding shape:", abstract_embeddings.shape)

100%|██████████| 157/157 [00:38<00:00,  4.11it/s]

Embedding shape: (2511, 768)


In [12]:
## Save
np.save("paper_ids.npy", np.array(paper_ids))
np.save("abstract_embeddings.npy", abstract_embeddings)

print("Saved paper_ids.npy and abstract_embeddings.npy")

Saved paper_ids.npy and abstract_embeddings.npy


In [7]:
import numpy as np
np.load('../../outputs/intermediate/paper_ids.npy'),

(array(['NOVEL_DIA_0', 'NOVEL_DIA_1', 'NOVEL_DIA_2', ..., 'SKG_SUM_218',
        'SKG_SUM_219', 'SKG_SUM_220'], shape=(2511,), dtype='<U12'),)

In [10]:
np.load('../../outputs/intermediate/abstract_embeddings.npy')

array([[ 0.05511691,  0.00364276,  0.07158133, ..., -0.2560189 ,
         0.07805625, -0.5801962 ],
       [ 0.05394249, -0.03677005, -0.00789899, ...,  0.02419077,
         0.06720246, -0.6158219 ],
       [-0.51626676,  0.19956142, -0.05647993, ...,  0.25275984,
        -0.23334438, -0.6393195 ],
       ...,
       [ 0.39936075, -0.08245266, -0.13111438, ..., -0.16277286,
         0.04630106, -0.7773472 ],
       [-0.03196314, -0.36695817, -0.12362581, ..., -0.27152818,
         0.04428685, -0.8497715 ],
       [ 0.15718073, -0.2899733 ,  0.04315058, ..., -0.08974425,
         0.16263288, -0.6295541 ]], shape=(2511, 768), dtype=float32)

In [11]:
## Diagonal should be ~1.
## Others < 1.
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(np.load('../../outputs/intermediate/abstract_embeddings.npy')[:5])
print(sim_matrix)

[[1.0000002  0.90534776 0.7392997  0.88580245 0.9082016 ]
 [0.90534776 1.0000002  0.7828707  0.9265648  0.8677758 ]
 [0.7392997  0.7828707  1.0000001  0.7357354  0.71477014]
 [0.88580245 0.9265648  0.7357354  1.0000001  0.8532336 ]
 [0.9082016  0.8677758  0.71477014 0.8532336  1.0000002 ]]
